# Quiet-Sun comparison: Sharma & Oberoi 2020, Figure 4

This notebook is for **looking** at the pictures, one step at a time.

We are not copying the journal figure. We are asking a science question:

> For a very quiet Sun on **3 December 2015**, 03:39–03:44 UT, can we remake the radio maps and the brightness in kelvin, and see what agrees with Sharma & Oberoi (2020) and what does not?

**What you need.** The conda environment `solar-burst-multiview`. Run the notebook from the `notebooks/` folder, or from the repo root. It only reads files already on disk; it does not talk to the cluster.

**How to use it.** Run the cells in order. Each code cell does one thing. The markdown cell above it says what you are looking at.


## 1. Load libraries

Nothing is plotted yet. This cell only imports tools.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import FITSFixedWarning
from IPython.display import Image, display, Markdown
from matplotlib.patches import Circle, Ellipse

warnings.filterwarnings("ignore", category=FITSFixedWarning)
%matplotlib inline
plt.rcParams["figure.dpi"] = 110


## 2. Find the project folder

The notebook works if you started Jupyter in `notebooks/` or in the repo root.


In [ ]:
here = Path.cwd()
if (here / "src" / "solarburst").exists():
    ROOT = here
elif (here.parent / "src" / "solarburst").exists():
    ROOT = here.parent
else:
    raise SystemExit("Cannot find src/solarburst. Open the notebook from the repo.")

src = ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from solarburst.maps import (
    frequency_hz,
    header_cell_arcsec,
    hpc_extent,
    jybeam_to_tb,
    load_casa_2d,
    solar_pixel,
)
from solarburst.mohan2017 import scale_image

print("project folder:", ROOT)


## 3. Check that the comparison files are on disk

If a line says **missing**, stop: that picture or table was not copied from the cluster yet.


In [ ]:
PATHS = {
    "figure 4 (our reconstruction)": ROOT / "figures" / "sharma2020" / "fig4.png",
    "Rohit 4-minute average": ROOT / "data" / "quiet2015" / "fig4_means.npz",
    "whole-Sun flux tables": ROOT / "data" / "quiet2015" / "paper_flux",
    "Haslam 1982 sky at this pointing": ROOT / "figures" / "sharma2020" / "haslam_sky.png",
    "WSClean 4-minute pictures": ROOT / "data" / "quiet2015" / "wsclean",
    "CASA half-second pictures": ROOT / "data" / "quiet2015" / "paper_casa",
    "WSClean half-second pictures": ROOT / "data" / "quiet2015" / "paper_wsclean",
}
for name, path in PATHS.items():
    ok = path.exists()
    extra = ""
    if path.is_dir() and ok:
        extra = f"  ({len(list(path.glob('*')))} files)"
    print(f"{'ok     ' if ok else 'MISSING'}  {name:32s}  {path}{extra}")


## 4. Look at Figure 4 first

This is **our** reconstruction of Sharma & Oberoi 2020 Figure 4, from Rohit’s already-made pictures.

- Top: radio Sun at 108, 162, 217 MHz (4-minute average).
- Middle: a model of thermal emission (FORWARD), not a telescope picture.
- Bottom: an extreme-ultraviolet picture of the Sun, with radio contours.

It is **not** a photocopy of the journal. Same observation, same style; the radio peaks do not match the paper caption exactly (see the next cells).


In [ ]:
fig4 = PATHS["figure 4 (our reconstruction)"]
display(Image(filename=str(fig4), width=900))
print(fig4)


## 5. Names we will keep using

| Word | Meaning |
|---|---|
| **Rohit** | Rohit Sharma, first author. His processed pictures live in his archive. |
| **CASA** | The picture-making program the 2020 paper used. |
| **WSClean** | A different picture-making program. We used it as a check. The paper did not. |
| **Half-second frame** | The telescope recorded a new picture every 0.5 s for four minutes (~570 pictures). A “frame” is one of those. A “4-minute average” is all of them stacked. |
| **Whole-Sun brightness** | One number per frequency: how bright the *entire* Sun is, measured from the recordings *without* making a map. The paper then recoulours the map so the disk adds up to that number. |
| **Million K** | Brightness temperature in millions of kelvin. Quiet corona is roughly 0.5–1.5 million K in this band. |

Four frequencies: **108, 162, 217, 240 MHz**. The paper’s Figure 4 top row uses the first three; 240 MHz is on the bottom row.


In [ ]:
BANDS = ("108", "162", "217", "240")
PAPER_PEAKS = {"108": 0.92, "162": 1.20, "217": 1.50, "240": None}  # million K; 240 not in the top row
RSUN = 958.0   # optical solar radius, arcsec
FOV = 2500.0   # half-width of the square we plot, arcsec
NPIX = 100     # crop size in pixels (50 arcsec per pixel → 5000 arcsec field)


## 6. A small drawing helper

One function, used by later cells: draw a radio map with 1- and 2-solar-radius rings, centred on the Sun. Run this once, then forget it.


In [ ]:
def draw_map(ax, tb, extent, beam, title, *, stretch_to_peak=False):
    """tb in kelvin. Brown rings are 1× and 2× the optical Sun, at (0, 0)."""
    tb_mk = np.asarray(tb, dtype=float) / 1e6
    peak = float(np.nanmax(tb_mk))
    if stretch_to_peak:
        cut, vmax = 0.12 * peak, max(peak, 1e-6)
    else:
        cut, vmax = 0.2, 1.7  # same as Figure 4
    shown = np.where(tb_mk > cut, tb_mk, np.nan)
    ax.imshow(shown, origin="lower", extent=extent, cmap="jet",
              vmin=0.0, vmax=vmax, interpolation="nearest")
    if peak > 0:
        ax.contour(tb_mk, levels=[f * peak for f in (0.70, 0.75, 0.80, 0.85, 0.90, 0.95)],
                   extent=extent, origin="lower", colors="black", linewidths=0.6)
    for r in (1.0, 2.0):
        ax.add_patch(Circle((0.0, 0.0), r * RSUN, fill=False, color="saddlebrown", lw=1.2))
    bmaj, bmin, bpa = beam
    ax.add_patch(Ellipse((-0.8 * FOV, -0.8 * FOV), bmin * 3600, bmaj * 3600,
                         angle=-bpa, fill=False, color="black", lw=1.0))
    ax.set_xlim(-FOV, FOV)
    ax.set_ylim(-FOV, FOV)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.text(0.03, 0.97, f"peak {peak:.2f} million K", transform=ax.transAxes,
            va="top", fontsize=9, color="white",
            bbox=dict(boxstyle="round,pad=0.15", fc="black", alpha=0.45, lw=0))
    return peak


## 7. Load Rohit’s 4-minute average (row 1)

These arrays are the time average of Rohit’s CASA pictures, already converted to kelvin with a **fudge factor 9.48**. That factor was measured against his 2022 stored cubes. It is **not** the 2020 paper’s brightness method. It only makes the maps look like his later work.

This is the data behind the top of `fig4.png`.


In [ ]:
packed = np.load(PATHS["Rohit 4-minute average"])
print("arrays in the file:", packed.files)
rohit = {}
chan = {"108": "084-085", "162": "125-126", "217": "169-170", "240": "187-188"}
freq_hz = {"108": 108.14e6, "162": 160.62e6, "217": 216.94e6, "240": 239.98e6}
for lab in BANDS:
    ch = chan[lab]
    hdr = fits.Header()
    hdr["BMAJ"] = float(packed[f"{ch}_bmaj"])
    hdr["BMIN"] = float(packed[f"{ch}_bmin"])
    hdr["BPA"] = float(packed[f"{ch}_bpa"])
    hdr["CDELT1"] = -50.0 / 3600.0
    hdr["CDELT2"] = 50.0 / 3600.0
    hdr["CRPIX1"] = (NPIX + 1) / 2.0
    hdr["CRPIX2"] = (NPIX + 1) / 2.0
    tb = np.asarray(packed[f"{ch}_tb"], dtype=float)
    # Undo 9.48 so later we can recolour from whole-Sun brightness.
    jy = tb / jybeam_to_tb(np.ones(1), freq_hz[lab], hdr["BMAJ"], hdr["BMIN"], scale=9.48)[0]
    rohit[lab] = {
        "tb": tb,
        "jy": jy,
        "header": hdr,
        "beam": (hdr["BMAJ"], hdr["BMIN"], hdr["BPA"]),
        "extent": np.asarray(packed[f"{ch}_extent"], dtype=float),
        "freq_hz": freq_hz[lab],
    }
    print(f"{lab} MHz  peak {tb.max()/1e6:.3f} million K  (paper caption {PAPER_PEAKS[lab]})")


## 8. Draw one Rohit map: 108 MHz

Look at: a radio disk, larger than the optical Sun (inner brown ring), and a brighter patch on the **west** (right in these plots). That is the quiet-Sun morphology in the paper.


In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5.2))
draw_map(ax, rohit["108"]["tb"], rohit["108"]["extent"], rohit["108"]["beam"],
         "Rohit, 4-minute average, 108 MHz", stretch_to_peak=False)
ax.set_xlabel("distance from Sun centre (arcsec)")
ax.set_ylabel("distance from Sun centre (arcsec)")
plt.show()


## 9. Draw all four Rohit frequencies

This is **row 1** of the comparison. Colour scale 0–1.7 million K, same as Figure 4. Compare the printed peak to the paper caption (108 / 162 / 217 MHz: 0.92 / 1.20 / 1.50).


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, lab in zip(axes, BANDS):
    draw_map(ax, rohit[lab]["tb"], rohit[lab]["extent"], rohit[lab]["beam"],
             f"{lab} MHz", stretch_to_peak=False)
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
fig.suptitle("Row 1 — Rohit’s 4-minute average (not a photocopy of the journal figure)")
plt.tight_layout()
plt.show()


## 10. Whole-Sun brightness: what it is

The 2020 paper does **not** set the kelvin scale from a calibrator star. It does two steps:

1. Measure how bright the **whole Sun** is from the shortest antenna pairings (Oberoi, Sharma & Rogers 2017). That is one number per frequency, in “solar flux units”.
2. Stretch the map so the disk adds up to that number, then convert to kelvin (Mohan & Oberoi 2017).

Row 2 is step 2 applied to **the same pixels as row 1**, using the whole-Sun numbers we measured on this observation.

The sky inside step 1 is now the Haslam et al. 1982 408 MHz map (the unfiltered LAMBDA file, the survey Oberoi 2017 actually cites), scaled with α = −2.55 and weighted by the MWA tile beam at **this** 3 December 2015 pointing. That morning the Sun sat about 17° from the Galactic centre, so the sky behind the Sun is much brighter than Oberoi’s 3 September 2013 high-latitude table. That was the missing piece, not a different map.


In [ ]:
flux_dir = PATHS["whole-Sun flux tables"]
s_odot = {}  # whole-Sun brightness, solar flux units
print(f"{'freq':>8s}  {'this sky (Haslam 1982)':>22s}  {'if we had used 2013 Table 1':>26s}  {'Oberoi 2013 day':>16s}")
check_2013 = {"108": 2.16, "162": 5.08, "217": 11.3, "240": 17.1}
for lab in BANDS:
    rec = json.loads((flux_dir / f"{lab}MHz_full.json").read_text())
    s_odot[lab] = rec["s_odot_sfu"]
    old = rec.get("table1_s_odot_sfu")
    print(
        f"{lab:>5s} MHz  {s_odot[lab]:22.2f}  {old:26.2f}  {check_2013[lab]:16.2f}"
    )
    print(" ", rec.get("t_sky_source", "")[:90])


## 11. Plot those whole-Sun numbers

Grey is **not** a target. It is Oberoi’s 2013 quiet day, a rough sanity check only. Blue is this 2015 morning, with the Haslam sky at *this* pointing. The December Sun sits near the Galactic plane, so the whole-Sun numbers went up; at 108 MHz that is what brings row 2 near the paper caption.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(len(BANDS))
ours = [s_odot[lab] for lab in BANDS]
ref = [check_2013[lab] for lab in BANDS]
ax.bar(x - 0.18, ours, 0.36, color="steelblue", label="3 Dec 2015 — this observation")
ax.bar(x + 0.18, ref, 0.36, color="0.75", label="3 Sep 2013 — different day, check only")
ax.set_xticks(x, [f"{lab} MHz" for lab in BANDS])
ax.set_ylabel("whole-Sun radio brightness (solar flux units)")
ax.set_title("The numbers used to recolour row 2")
ax.legend(frameon=False)
ax.set_ylim(0, 20)
for i, v in enumerate(ours):
    ax.text(i - 0.18, v + 0.25, f"{v:.1f}", ha="center")
plt.show()


## 11b. The sky map behind the Sun

This is the Haslam et al. 1982 408 MHz survey — the same map Oberoi 2017 used — shown as the sky above the horizon at 03:42 UT on 3 December 2015, scaled to 108 MHz. The yellow dot is the Sun. It sits on a bright part of the Milky Way, which is why the whole-Sun numbers went up relative to the 2013 table.

The white contours are the MWA tile beam (90 %, 9 %, 0.9 %). That beam is a 4×4 array factor with the published tile geometry, not Oberoi’s unpublished FEKO simulation.


In [ ]:
haslam_png = PATHS["Haslam 1982 sky at this pointing"]
if haslam_png.exists():
    display(Image(filename=str(haslam_png), width=900))
    print(haslam_png)
else:
    print("Haslam sky PNG is not on disk; run scripts/sharma2020_haslam_tsky.py")


## 12. Recolour Rohit’s maps with those numbers (row 2)

Same picture, new colour. The stretch factor is “how far off was the map’s own disk sum from the whole-Sun number”.


In [ ]:
paper_scaled = {}
print(f"{'freq':>8s}  {'row 1 peak':>12s}  {'row 2 peak':>12s}  {'stretch':>10s}")
for lab in BANDS:
    rec = scale_image(
        rohit[lab]["jy"],
        rohit[lab]["header"],
        s_odot[lab] * 1e4,   # solar flux units → jansky
        nsig=5.0,
        freq_hz=rohit[lab]["freq_hz"],
    )
    rec["beam"] = rohit[lab]["beam"]
    rec["extent"] = rohit[lab]["extent"]
    paper_scaled[lab] = rec
    print(f"{lab:>5s} MHz  {rohit[lab]['tb'].max()/1e6:12.3f}  {rec['tb_K'].max()/1e6:12.3f}  {rec['f_nu']:10.2f}")


## 13. Before / after at 108 MHz

Left = row 1 (fudge factor 9.48). Right = row 2 (whole-Sun number). The **shape cannot change**. Only the colour numbers change.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
draw_map(axes[0], rohit["108"]["tb"], rohit["108"]["extent"], rohit["108"]["beam"],
         "Row 1 — fudge factor 9.48", stretch_to_peak=False)
draw_map(axes[1], paper_scaled["108"]["tb_K"], paper_scaled["108"]["extent"],
         paper_scaled["108"]["beam"], "Row 2 — whole-Sun brightness", stretch_to_peak=True)
for ax in axes:
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
plt.tight_layout()
plt.show()


## 14. Why rows 3–5 looked shifted

The radio Sun is **not** in the middle of the large picture file. The brightest pixel is the western patch, a bit south of disk centre.

If you crop on the brightest pixel, you put that patch at (0, 0) and the brown rings (drawn at the Sun) look wrong. That was a display bug. The next cell shows both crops at 108 MHz. **Always crop on the Sun.**


In [ ]:
data, hdr = load_casa_2d(PATHS["WSClean 4-minute pictures"] / "108MHz-image.fits")
sx, sy = solar_pixel(hdr)
py, px = np.unravel_index(int(np.nanargmax(data)), data.shape)
print(f"image size {data.shape}")
print(f"Sun pixel (from the time in the header): x={sx:.1f}, y={sy:.1f}")
print(f"brightest pixel:                         x={px:.0f}, y={py:.0f}")
print(f"offset: {px - sx:.0f} pixels in x, {py - sy:.0f} pixels in y  (1 pixel = 50 arcsec)")


def crop_at(data, hdr, cx, cy, npix=NPIX):
    x0 = int(round(cx - (npix - 1) / 2.0))
    y0 = int(round(cy - (npix - 1) / 2.0))
    crop = data[y0:y0 + npix, x0:x0 + npix]
    extent = hpc_extent(npix, npix, cell_arcsec=header_cell_arcsec(hdr),
                        sun_xy=(sx, sy), origin=(x0, y0))
    beam = (float(hdr["BMAJ"]), float(hdr["BMIN"]), float(hdr.get("BPA", 0.0)))
    tb = jybeam_to_tb(crop, frequency_hz(hdr), beam[0], beam[1], scale=1.0)
    return tb, extent, beam

tb_wrong, ext_wrong, beam = crop_at(data, hdr, px, py)
tb_right, ext_right, beam = crop_at(data, hdr, sx, sy)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
draw_map(axes[0], tb_wrong, ext_wrong, beam, "Cropped on the bright patch — WRONG", stretch_to_peak=True)
draw_map(axes[1], tb_right, ext_right, beam, "Cropped on the Sun — right", stretch_to_peak=True)
for ax in axes:
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
plt.tight_layout()
plt.show()


## 15. Load a remade picture, cropped on the Sun

Helper used for WSClean and CASA files. Brightness here is kelvin from the Rayleigh–Jeans formula with **no** fudge factor and **no** whole-Sun stretch yet. Peaks will look low (~0.08 million K). Look at **shape**, not the colour numbers.


In [ ]:
def load_picture(path: Path) -> dict:
    data, hdr = load_casa_2d(path)
    sx, sy = solar_pixel(hdr)
    x0 = int(round(sx - (NPIX - 1) / 2.0))
    y0 = int(round(sy - (NPIX - 1) / 2.0))
    crop = data[y0:y0 + NPIX, x0:x0 + NPIX]
    extent = hpc_extent(NPIX, NPIX, cell_arcsec=header_cell_arcsec(hdr),
                        sun_xy=(sx, sy), origin=(x0, y0))
    beam = (float(hdr["BMAJ"]), float(hdr["BMIN"]), float(hdr.get("BPA", 0.0)))
    tb = jybeam_to_tb(crop, frequency_hz(hdr), beam[0], beam[1], scale=1.0)
    return {"tb": tb, "beam": beam, "extent": extent, "path": path}

wsclean_4min = {
    lab: load_picture(PATHS["WSClean 4-minute pictures"] / f"{lab}MHz-image.fits")
    for lab in BANDS
}
print("loaded 4-minute WSClean pictures")
for lab in BANDS:
    print(f"  {lab} MHz  peak {wsclean_4min[lab]['tb'].max()/1e6:.3f} million K")


## 16. Row 3 — we remade a 4-minute picture with WSClean

WSClean is **our** picture-making program. The 2020 paper used CASA, not WSClean. This row is a check: do we get the same quiet-Sun shape from the visibilities?


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, lab in zip(axes, BANDS):
    m = wsclean_4min[lab]
    draw_map(ax, m["tb"], m["extent"], m["beam"], f"{lab} MHz", stretch_to_peak=True)
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
fig.suptitle("Row 3 — 4 minutes stacked, WSClean (not the paper’s program)")
plt.tight_layout()
plt.show()


## 17. Row 4 — one half-second, CASA (the paper’s program)

The paper made one map every 0.5 s, then averaged. Here we remake **one** of those moments (the middle of the scan, 03:42:04 UT) with CASA, using the paper’s picture size (1024 pixels, 50 arcsec, 2500 clean iterations).

This is not the 4-minute average. It should look noisier, but the disk and western patch should still be there.


In [ ]:
casa_half = {
    lab: load_picture(PATHS["CASA half-second pictures"] / f"{lab}MHz_mid.fits")
    for lab in BANDS
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, lab in zip(axes, BANDS):
    m = casa_half[lab]
    draw_map(ax, m["tb"], m["extent"], m["beam"], f"{lab} MHz", stretch_to_peak=True)
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
fig.suptitle("Row 4 — one half-second snapshot, CASA (the paper’s program)")
plt.tight_layout()
plt.show()


## 18. Row 5 — the same half-second, WSClean

Same moment as row 4, different program. Compare 4 and 5: if they agree, the choice of program is not the main difference. If they disagree, the program matters.


In [ ]:
wsclean_half = {
    lab: load_picture(PATHS["WSClean half-second pictures"] / f"{lab}MHz_mid-image.fits")
    for lab in BANDS
}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, lab in zip(axes, BANDS):
    m = wsclean_half[lab]
    draw_map(ax, m["tb"], m["extent"], m["beam"], f"{lab} MHz", stretch_to_peak=True)
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
fig.suptitle("Row 5 — same half-second as row 4, WSClean")
plt.tight_layout()
plt.show()


## 19. One frequency, all five versions

108 MHz, side by side. Ask: is the western patch on the west? Does the disk sit in the inner brown ring? Which rows share a colour scale (only row 1 uses 0–1.7 million K; the others stretch to their own peak so you can see the shape)?


In [ ]:
lab = "108"
panels = [
    ("1. Rohit, 4 min", rohit[lab]["tb"], rohit[lab]["extent"], rohit[lab]["beam"], False),
    ("2. Same, whole-Sun colour", paper_scaled[lab]["tb_K"], paper_scaled[lab]["extent"], paper_scaled[lab]["beam"], True),
    ("3. WSClean, 4 min", wsclean_4min[lab]["tb"], wsclean_4min[lab]["extent"], wsclean_4min[lab]["beam"], True),
    ("4. CASA, 0.5 s", casa_half[lab]["tb"], casa_half[lab]["extent"], casa_half[lab]["beam"], True),
    ("5. WSClean, 0.5 s", wsclean_half[lab]["tb"], wsclean_half[lab]["extent"], wsclean_half[lab]["beam"], True),
]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (title, tb, extent, beam, stretch) in zip(axes, panels):
    draw_map(ax, tb, extent, beam, title, stretch_to_peak=stretch)
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
fig.suptitle("108 MHz — five versions of the same quiet Sun")
plt.tight_layout()
plt.show()


## 20. Change `lab` and rerun

Set `lab` to `"162"`, `"217"`, or `"240"` and run the cell. Nothing else to change.


In [ ]:
lab = "162"  # try 108, 162, 217, 240
panels = [
    ("1. Rohit, 4 min", rohit[lab]["tb"], rohit[lab]["extent"], rohit[lab]["beam"], False),
    ("2. Same, whole-Sun colour", paper_scaled[lab]["tb_K"], paper_scaled[lab]["extent"], paper_scaled[lab]["beam"], True),
    ("3. WSClean, 4 min", wsclean_4min[lab]["tb"], wsclean_4min[lab]["extent"], wsclean_4min[lab]["beam"], True),
    ("4. CASA, 0.5 s", casa_half[lab]["tb"], casa_half[lab]["extent"], casa_half[lab]["beam"], True),
    ("5. WSClean, 0.5 s", wsclean_half[lab]["tb"], wsclean_half[lab]["extent"], wsclean_half[lab]["beam"], True),
]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (title, tb, extent, beam, stretch) in zip(axes, panels):
    draw_map(ax, tb, extent, beam, title, stretch_to_peak=stretch)
    ax.set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
fig.suptitle(f"{lab} MHz — five versions")
plt.tight_layout()
plt.show()


## 21. Peak brightness table

Row 1 should be compared with the paper caption. Rows 3–5 are still without the whole-Sun recolour, so their kelvin peaks are not yet on the paper’s scale.


In [ ]:
print(f"{'freq':>8s}  {'paper':>8s}  {'row 1':>8s}  {'row 2':>8s}  {'row 3':>8s}  {'row 4':>8s}  {'row 5':>8s}")
print("          caption   Rohit    whole-Sun WSClean4  CASA 0.5  WSClean 0.5")
for lab in BANDS:
    cap = PAPER_PEAKS[lab]
    cap_s = f"{cap:8.2f}" if cap else "     —"
    print(
        f"{lab:>5s} MHz {cap_s}  {rohit[lab]['tb'].max()/1e6:8.3f}  "
        f"{paper_scaled[lab]['tb_K'].max()/1e6:8.3f}  {wsclean_4min[lab]['tb'].max()/1e6:8.3f}  "
        f"{casa_half[lab]['tb'].max()/1e6:8.3f}  {wsclean_half[lab]['tb'].max()/1e6:8.3f}"
    )


## 22. What to take away

**Solid**

- Row 1 has the paper’s quiet-Sun shape: a disk plus a western bright patch, larger radio Sun at 108 MHz.
- Rows 3–5, once centred on the Sun, show the same shape. Remaking the pictures from the visibilities did not invent a different Sun.
- CASA (row 4) and WSClean (row 5) on the same half-second look alike, so the program is not the main story.

**Not the journal figure**

- Row 1 peaks are lower than the caption (0.79 / 0.77 / 0.77 vs 0.92 / 1.20 / 1.50 million K).
- Row 2 is the 2020 brightness method with the Haslam 1982 sky at this pointing. At 108 MHz the peak should now sit near the caption (0.92 million K); 162 and 217 MHz may still run low.
- Rows 3–5 are still on a raw kelvin scale (no whole-Sun stretch). Compare shape, not the printed peak.
- The tile beam in the sky integral is a 4×4 array factor, not Oberoi’s unpublished FEKO patterns. Solid angle matches Table 1; far sidelobes at 103 MHz on the 2013 pointing are still about 2× high.

**Worth trying next**

- Put the whole-Sun number onto rows 3–5 and see whether the kelvin peaks meet row 2.
- Average many half-second CASA frames, as the paper did, instead of one frame.
- The FEKO embedded beam, if it ever turns up, replaces the 4×4 array factor.

The comparison grid PNG (all rows, all frequencies) is `figures/sharma2020/compare_maps.png` if you want it in one image.


In [ ]:
grid = ROOT / "figures" / "sharma2020" / "compare_maps.png"
if grid.exists():
    display(Image(filename=str(grid), width=900))
    print(grid)
else:
    print("comparison grid PNG is not on disk")
